In [1]:
import torch
from torch import nn

In [2]:
class CustomLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        """
        自定义LayerNorm实现
        
        参数:
        - normalized_shape: 归一化的特征维度
        - eps: 数值稳定性小常数，防止除以零
        - elementwise_affine: 是否使用可学习的缩放(γ)和平移(β)参数
        """
        super().__init__()
        
        # 检查normalized_shape是int还是tuple
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        
        if self.elementwise_affine:
            # 可学习的缩放参数γ，初始化为1
            self.gamma = nn.Parameter(torch.ones(*self.normalized_shape))
            # 可学习的平移参数β，初始化为0
            self.beta = nn.Parameter(torch.zeros(*self.normalized_shape))
        else:
            # 不使用可学习参数
            self.register_parameter('gamma', None)
            self.register_parameter('beta', None)
    
    def forward(self, x):
        # 计算均值和方差
        mean = x.mean(dim=-1, keepdim=True)
        #有偏估计 (分母n)
        # 方差计算公式：
        # 有偏公式 (1/n)((x1-mean)**2+(x2-mean)**2+...+(xn-mean)**2)
        # 无偏公式 (1/(n-1))((x1-mean)**2+(x2-mean)**2+...+(xn-mean)**2)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        
        # 归一化公式
        x_normalized = (x - mean) / torch.sqrt(variance + self.eps)
        
        # 应用可学习的参数
        if self.elementwise_affine:
            output = self.gamma * x_normalized + self.beta
        else:
            output = x_normalized
            
        return output

In [4]:
# 创建输入数据: (batch_size, seq_len, features)
batch_size, seq_len, features = 16, 10, 512
x = torch.randn(batch_size, seq_len, features)

print("输入形状:", x.shape)
print("输出形状:", x.shape)
print("输出均值:", x.mean(dim=-1))
print("输出方差:", x.var(dim=-1, unbiased=False))
# 实例化自定义层归一化模块
custom_norm = CustomLayerNorm(features)

# 前向传播
output = custom_norm(x)

print("输入形状:", x.shape)
print("输出形状:", output.shape)
print("输出均值:", output.mean(dim=-1))
print("输出方差:", output.var(dim=-1, unbiased=False))

输入形状: torch.Size([16, 10, 512])
输出形状: torch.Size([16, 10, 512])
输出均值: tensor([[-1.8626e-08,  1.1642e-08,  3.7253e-09, -9.3132e-10, -3.7253e-09,
          3.7253e-09,  2.7940e-09,  2.7940e-09, -3.2596e-09,  2.3283e-09],
        [ 7.4506e-09, -3.7253e-09,  5.5879e-09, -1.0245e-08,  1.1176e-08,
         -1.8626e-09, -3.7253e-09,  6.5193e-09,  9.3132e-10,  1.1642e-08],
        [ 1.3970e-08, -1.3970e-09,  5.5879e-09, -1.3039e-08, -5.5879e-09,
         -2.7940e-09,  4.6566e-09, -9.3132e-09,  7.4506e-09,  7.4506e-09],
        [ 4.6566e-09,  1.0012e-08,  9.3132e-09, -1.3039e-08,  1.1176e-08,
         -5.5879e-09,  1.8626e-09, -7.4506e-09,  1.8626e-09, -1.7695e-08],
        [-3.7253e-09,  1.1176e-08,  8.8476e-09,  7.4506e-09, -3.7253e-09,
          6.0536e-09,  5.5879e-09,  1.4901e-08, -7.4506e-09,  6.9849e-10],
        [ 5.5879e-09, -8.3819e-09, -7.4506e-09, -1.5832e-08,  1.8626e-09,
         -3.2596e-09,  5.5879e-09,  0.0000e+00, -9.3132e-09, -1.1176e-08],
        [ 5.8208e-09,  0.0000e+00, -

In [6]:
## softmax 实现
# 多分类网络输出层示例
def softmax_torch(x, dim=-1):
    """PyTorch实现的softmax函数
    
    参数:
        x - 输入张量
        dim - 计算softmax的维度
        
    返回:
        softmax激活后的张量
    """
    # 数值稳定性处理
    max_vals = torch.max(x, dim=dim, keepdim=True).values
    e_x = torch.exp(x - max_vals)
    
    return e_x / torch.sum(e_x, dim=dim, keepdim=True)


In [8]:
seq_len =  10
x = torch.randn(seq_len)

result = softmax_torch(x)
print(f"result is {result}")
print(f"sum of result is {sum(result)}")

result is tensor([0.2177, 0.2169, 0.0179, 0.0287, 0.1962, 0.0557, 0.0332, 0.1302, 0.0759,
        0.0276])
sum of result is 0.9999999403953552


In [9]:
def softmax_temperature(x, temperature=1.0, dim=-1):
    """带有温度参数的softmax
    
    温度参数作用:
        temperature > 1.0: 平滑分布 (增加熵)
        temperature < 1.0: 锐化分布 (降低熵)
    """
    e_x = torch.exp(x / temperature)
    return e_x / torch.sum(e_x, dim=dim, keepdim=True)

In [11]:
result = softmax_temperature(x, 2)
print(f"result is {result}")
print(f"sum of result is {sum(result)}")

result is tensor([0.1609, 0.1607, 0.0461, 0.0584, 0.1528, 0.0814, 0.0628, 0.1245, 0.0950,
        0.0573])
sum of result is 1.0


In [14]:
result = softmax_temperature(x, 0.2)
print(f"result is {result}")
print(f"sum of result is {sum(result)}")

result is tensor([3.7594e-01, 3.6952e-01, 1.4127e-06, 1.4889e-05, 2.2337e-01, 4.1335e-04,
        3.0978e-05, 2.8753e-02, 1.9418e-03, 1.2388e-05])
sum of result is 1.0


In [16]:
def batched_softmax(x):
    """处理批量输入的softmax
    
    输入形状: (batch_size, num_classes)
    输出形状: (batch_size, num_classes)
    """
    max_vals, _ = torch.max(x, dim=1, keepdim=True)
    e_x = torch.exp(x - max_vals)
    return e_x / torch.sum(e_x, dim=1, keepdim=True)

In [19]:
batch_size, seq_len = 16, 10
x = torch.randn(batch_size, seq_len)

result = batched_softmax(x)
print(f"result is {result}")
print(f"sum of result is {torch.sum(result, -1)}")

result is tensor([[0.0249, 0.0963, 0.1173, 0.0254, 0.0398, 0.0308, 0.0253, 0.0864, 0.1333,
         0.4205],
        [0.0721, 0.0575, 0.0718, 0.0946, 0.0730, 0.0700, 0.0816, 0.1309, 0.3122,
         0.0362],
        [0.1352, 0.0356, 0.0904, 0.0697, 0.0597, 0.1568, 0.0579, 0.0479, 0.2534,
         0.0934],
        [0.0611, 0.2840, 0.0757, 0.1075, 0.2521, 0.0165, 0.0937, 0.0456, 0.0142,
         0.0497],
        [0.0838, 0.1366, 0.0986, 0.0614, 0.1189, 0.1207, 0.1235, 0.1177, 0.0437,
         0.0950],
        [0.0544, 0.0719, 0.0155, 0.5224, 0.0729, 0.1248, 0.0397, 0.0166, 0.0483,
         0.0335],
        [0.0512, 0.1723, 0.0225, 0.0638, 0.0443, 0.1301, 0.0875, 0.3488, 0.0371,
         0.0424],
        [0.1368, 0.0213, 0.0812, 0.0967, 0.1367, 0.1791, 0.1165, 0.0258, 0.1291,
         0.0768],
        [0.2714, 0.0150, 0.1086, 0.0988, 0.1231, 0.1539, 0.0196, 0.0335, 0.0096,
         0.1665],
        [0.0134, 0.0419, 0.0631, 0.1270, 0.2279, 0.0326, 0.1060, 0.2378, 0.0554,
         0.0950],
